# 📄 Contract Analysis System — Phase 1
## Fine-tuning Qwen2.5-3B-Instruct với QLoRA trên CUAD Dataset

---

### ⚙️ Kaggle Setup (làm trước khi chạy)

1. **Accelerator**: `Settings → Accelerator → GPU P100` hoặc `T4 x2`  
2. **Internet**: `Settings → Internet → On` *(bắt buộc để tải model + dataset)*  
3. **HuggingFace Token** *(nếu cần model private)*:
   - Vào `Add-ons → Secrets → Add New Secret`
   - Name: `HF_TOKEN` | Value: token từ huggingface.co/settings/tokens

---

### 🗺️ Kiến trúc
| | |
|---|---|
| **Base Model** | `Qwen/Qwen2.5-3B-Instruct` |
| **Fine-tuning** | QLoRA (4-bit NF4 + LoRA r=16) |
| **Dataset** | CUAD (Contract Understanding Atticus Dataset) |
| **Runtime** | Kaggle P100 16GB hoặc T4 x2 |
| **Output** | `/kaggle/working/contract_lora_adapter/` |

### 📤 Output format
```json
{"category": "confidentiality", "summary": "Employee must keep company information confidential."}
```
**Categories:** `salary` · `payment` · `confidentiality` · `liability` · `termination` · `insurance` · `dispute_resolution` · `other`

---
## 📦 Cell 1 — Cài đặt thư viện

Kaggle đã pre-install `transformers`, `datasets`, `accelerate`, `scipy`.  
Chỉ cần cài thêm `peft`, `trl`, `bitsandbytes` với phiên bản tương thích.

In [1]:
# ============================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN
# ============================================================
# Kaggle đã có sẵn: transformers, datasets, accelerate, scipy
# Cần cài thêm: peft, trl, bitsandbytes

import subprocess, sys

packages = [
    "peft==0.12.0",
    "trl==0.12.0",
    "bitsandbytes==0.46.1",
    "einops",
]

for pkg in packages:
    print(f"📦 Installing {pkg}...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        check=True
    )

print("\n✅ Cài đặt hoàn tất!")

📦 Installing peft==0.12.0...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

📦 Installing trl==0.12.0...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 9.2 MB/s eta 0:00:00
📦 Installing bitsandbytes==0.46.1...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 22.4 MB/s eta 0:00:00
📦 Installing einops...

✅ Cài đặt hoàn tất!


---
## 🔧 Cell 2 — Import và kiểm tra GPU

Kiểm tra GPU Kaggle đang dùng:
- **P100**: 16GB GDDR5 — tốt nhất cho notebook này
- **T4 x2**: 2 × 16GB — cần `device_map="auto"` để dùng cả 2

> Nếu thấy `No GPU found` → vào Settings → Accelerator → bật GPU

In [2]:
# ============================================================
# CELL 2: IMPORT VÀ KIỂM TRA GPU
# ============================================================

import os, json, re, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
from collections import Counter
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig, TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)
from trl import SFTTrainer, SFTConfig

# ── Kiểm tra GPU ────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("❌ Không tìm thấy GPU! Vào Settings → Accelerator → GPU")

n_gpus = torch.cuda.device_count()
print(f"🖥️  Số GPU: {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    vram  = props.total_memory / 1e9
    print(f"   GPU {i}: {props.name} | VRAM: {vram:.1f} GB")

# Phát hiện loại GPU để điều chỉnh config
gpu_name = torch.cuda.get_device_name(0).lower()
IS_P100  = "p100" in gpu_name
IS_T4    = "t4"   in gpu_name

print(f"\n✅ Detected: {'P100' if IS_P100 else 'T4' if IS_T4 else gpu_name.upper()}")
print(f"✅ PyTorch: {torch.__version__}")

# Kaggle working directory
WORK_DIR = "/kaggle/working"
print(f"✅ Working dir: {WORK_DIR}")

🖥️  Số GPU: 2
   GPU 0: Tesla T4 | VRAM: 15.6 GB
   GPU 1: Tesla T4 | VRAM: 15.6 GB

✅ Detected: T4
✅ PyTorch: 2.10.0+cu128
✅ Working dir: /kaggle/working


---
## 🔑 Cell 3 — HuggingFace Login (nếu cần)

Qwen2.5-3B-Instruct là public model, **không bắt buộc** phải login.  
Bỏ comment nếu gặp lỗi rate-limit hoặc dùng model private.

> Token lấy từ: huggingface.co → Settings → Access Tokens  
> Lưu vào Kaggle Secrets với key `HF_TOKEN`

In [3]:
# ============================================================
# CELL 3: HUGGINGFACE LOGIN (TÙY CHỌN)
# ============================================================

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")

    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("✅ Đăng nhập HuggingFace thành công!")

except Exception as e:
    print(f"ℹ️  Bỏ qua HuggingFace login ({type(e).__name__}).")
    print("   Model Qwen2.5-3B-Instruct là public — không bắt buộc login.")

ℹ️  Bỏ qua HuggingFace login (BackendError).
   Model Qwen2.5-3B-Instruct là public — không bắt buộc login.


---
## 📂 Cell 4 — Tải và khám phá CUAD Dataset

**CUAD** (Contract Understanding Atticus Dataset):
- ~13,000 QA samples từ 510 hợp đồng thực tế
- 41 loại điều khoản do luật sư gán nhãn
- Format: `{context, question, answers: {text, answer_start}}`

Chúng ta dùng `context` làm input và `question` để xác định category.

In [4]:
print("Dang tai CUAD dataset (parquet)...")
# Dung parquet branch (refs/convert/parquet) thay vi loading script
# cuad-qa.py khong con duoc ho tro tu datasets >= 2.20
BASE = "https://huggingface.co/datasets/theatticusproject/cuad-qa/resolve/refs%2Fconvert%2Fparquet"
raw_dataset = load_dataset(
    "parquet",
    data_files={
        "train": [
            f"{BASE}/default/train/0000.parquet",
            f"{BASE}/default/train/0001.parquet",
            f"{BASE}/default/train/0002.parquet",
        ],
        "test": [f"{BASE}/default/test/0000.parquet"],
    },
)
print(raw_dataset)


Dang tai CUAD dataset (parquet)...


default/train/0000.parquet:   0%|          | 0.00/4.32M [00:00<?, ?B/s]

default/train/0001.parquet:   0%|          | 0.00/4.08M [00:00<?, ?B/s]

default/train/0002.parquet:   0%|          | 0.00/3.69M [00:00<?, ?B/s]

default/test/0000.parquet:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 22450
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 4182
    })
})


---
## 🗺️ Cell 5 — Mapping CUAD labels → 8 Categories chuẩn

CUAD có 41 câu hỏi. Chúng ta map về 8 category bằng **keyword matching** trên nội dung câu hỏi:

| Category | Keywords CUAD |
|----------|---------------|
| `salary` | salary, wage, compensation, bonus |
| `payment` | payment, fee, price, revenue, royalty |
| `confidentiality` | confidential, non-disclosure, trade secret |
| `liability` | liability, indemnif, damages |
| `termination` | terminat, expir, cancel, renewal |
| `insurance` | insurance, coverage, insurer |
| `dispute_resolution` | dispute, arbitration, governing law, jurisdiction |
| `other` | *(không khớp category nào)* |

In [5]:
# ============================================================
# CELL 5: MAPPING CUAD LABELS → CATEGORIES CHUẨN
# ============================================================

CATEGORY_KEYWORDS = {
    "salary": [
        "salary", "wage", "compensation", "remuneration", "bonus",
        "annual base salary", "base salary",
    ],
    "payment": [
        "payment", "fee", "price", "revenue share", "royalty",
        "revenue", "pricing", "invoice", "billing", "cost",
    ],
    "confidentiality": [
        "confidential", "non-disclosure", "nda", "proprietary",
        "trade secret", "disclosure",
    ],
    "liability": [
        "liability", "indemnif", "limitation of liability",
        "damages", "cap on liability",
    ],
    "termination": [
        "terminat", "expir", "cancel", "renewal",
        "notice period", "end date", "term of agreement",
    ],
    "insurance": [
        "insurance", "coverage", "insurer", "policy",
    ],
    "dispute_resolution": [
        "dispute", "arbitration", "governing law", "jurisdiction",
        "litigation", "mediation", "court",
    ],
}

def map_question_to_category(question: str) -> str:
    q = question.lower()
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(kw in q for kw in keywords):
            return category
    return "other"

# Xem phân phối category trên toàn dataset
all_categories = [map_question_to_category(s["question"]) for s in raw_dataset["train"]]
cat_counts = Counter(all_categories)

print("📊 Phân phối category trong CUAD (trước khi filter):")
total = sum(cat_counts.values())
for cat, count in sorted(cat_counts.items(), key=lambda x: -x[1]):
    pct = count / total * 100
    bar = "█" * int(pct / 2)
    print(f"   {cat:20s}: {count:5,d} ({pct:4.1f}%) {bar}")

# Test mapping
print("\n🧪 Test mapping:")
tests = [
    "Does the clause mention confidentiality obligations?",
    "What are the termination provisions?",
    "What is the annual base salary?",
    "What governing law applies?",
    "Is there an insurance requirement?",
]
for q in tests:
    print(f"   [{map_question_to_category(q):20s}] {q}")

📊 Phân phối category trong CUAD (trước khi filter):
   other               : 15,250 (67.9%) █████████████████████████████████
   payment             : 2,589 (11.5%) █████
   termination         : 2,274 (10.1%) █████
   liability           :   731 ( 3.3%) █
   insurance           :   717 ( 3.2%) █
   confidentiality     :   461 ( 2.1%) █
   dispute_resolution  :   428 ( 1.9%) 

🧪 Test mapping:
   [confidentiality     ] Does the clause mention confidentiality obligations?
   [termination         ] What are the termination provisions?
   [salary              ] What is the annual base salary?
   [dispute_resolution  ] What governing law applies?
   [insurance           ] Is there an insurance requirement?


---
## 🔄 Cell 6 — Chuyển đổi CUAD → Instruction Tuning Format

Chuyển từ QA format sang **Qwen2.5 Chat Template**:

```
<|im_start|>system
You are a legal contract analysis expert...
<|im_end|>
<|im_start|>user
Analyze the following contract clause...
<|im_end|>
<|im_start|>assistant
{"category": "confidentiality", "summary": "..."}
<|im_end|>
```

Chỉ giữ samples có `answers` (không rỗng) để đảm bảo chất lượng.

In [6]:
# ============================================================
# CELL 6: CHUYỂN ĐỔI DỮ LIỆU → INSTRUCTION TUNING FORMAT
# ============================================================

SYSTEM_PROMPT = (
    "You are a legal contract analysis expert. "
    "Analyze the given contract clause and return a JSON object with:\n"
    "- \"category\": one of [salary, payment, confidentiality, liability, "
    "termination, insurance, dispute_resolution, other]\n"
    "- \"summary\": a concise 1-2 sentence summary of the clause\n\n"
    "Return ONLY valid JSON, no additional text."
)

MAX_CONTEXT_CHARS = 1600  # ~400 tokens, an toàn cho P100/T4


def make_summary(answer_text: str, context: str) -> str:
    """Tạo summary từ answer text của CUAD."""
    text = answer_text.strip()
    if len(text) > 20:
        return text[:200] + "..." if len(text) > 200 else text
    # Fallback: câu đầu tiên của context
    first = context.split(".")[0].strip()
    return first + "."


def build_chat_text(context: str, category: str, summary: str) -> str:
    """Tạo full chat text theo Qwen2.5 format."""
    if len(context) > MAX_CONTEXT_CHARS:
        context = context[:MAX_CONTEXT_CHARS] + "..."

    user_msg = (
        "Analyze the following contract clause and return a JSON response:\n\n"
        f"CONTRACT CLAUSE:\n{context.strip()}"
    )
    output_json = json.dumps({"category": category, "summary": summary}, ensure_ascii=False)

    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n{output_json}<|im_end|>"
    )


def convert_cuad(dataset_split):
    """Chuyển đổi một split của CUAD sang instruction format."""
    converted, skipped = [], 0

    for sample in dataset_split:
        context  = sample.get("context", "").strip()
        question = sample.get("question", "").strip()
        answers  = sample.get("answers", {})
        texts    = answers.get("text", [])

        if not texts or not context:
            skipped += 1
            continue

        category = map_question_to_category(question)
        summary  = make_summary(texts[0], context)
        text     = build_chat_text(context, category, summary)

        converted.append({"text": text, "category": category})

    return converted, skipped


print("⏳ Chuyển đổi CUAD train split...")
all_data, n_skipped = convert_cuad(raw_dataset["train"])
print(f"   ✅ Converted: {len(all_data):,}")
print(f"   ⏭️  Skipped:   {n_skipped:,}")

# Preview
print("\n🔍 Preview sample:")
print("-" * 65)
print(all_data[0]["text"][:600] + "\n...")
print("-" * 65)

⏳ Chuyển đổi CUAD train split...
   ✅ Converted: 11,180
   ⏭️  Skipped:   11,270

🔍 Preview sample:
-----------------------------------------------------------------
<|im_start|>system
You are a legal contract analysis expert. Analyze the given contract clause and return a JSON object with:
- "category": one of [salary, payment, confidentiality, liability, termination, insurance, dispute_resolution, other]
- "summary": a concise 1-2 sentence summary of the clause

Return ONLY valid JSON, no additional text.<|im_end|>
<|im_start|>user
Analyze the following contract clause and return a JSON response:

CONTRACT CLAUSE:
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and be
...
-----------------------------------------------------------------


---
## ⚖️ Cell 7 — Cân bằng dataset và tạo Train/Validation split

Dataset CUAD mất cân bằng nặng. Thực hiện:
1. **Cap** tối đa 400 samples/category
2. **Shuffle** ngẫu nhiên (seed cố định để tái hiện)
3. **Split** 90% train / 10% validation

In [7]:
# ============================================================
# CELL 7: CÂN BẰNG DATASET VÀ TRAIN/VAL SPLIT
# ============================================================

random.seed(42)
np.random.seed(42)

MAX_PER_CLASS = 400  # P100 có 16GB nên thoải mái hơn T4

# Nhóm theo category
by_cat: dict = {}
for item in all_data:
    by_cat.setdefault(item["category"], []).append(item)

# Capping + shuffle
balanced = []
print(f"📊 Cân bằng dataset (max {MAX_PER_CLASS}/class):")
for cat in sorted(by_cat):
    items = by_cat[cat]
    random.shuffle(items)
    selected = items[:MAX_PER_CLASS]
    balanced.extend(selected)
    bar = "█" * (len(selected) // 20)
    print(f"   {cat:20s}: {len(selected):4d}  {bar}")

random.shuffle(balanced)
print(f"\n   Total: {len(balanced):,} samples")

# HuggingFace Dataset
hf_ds = Dataset.from_list([{"text": d["text"]} for d in balanced])

# Train / Val split
splits       = hf_ds.train_test_split(test_size=0.1, seed=42)
train_ds     = splits["train"]
val_ds       = splits["test"]

print(f"\n   🏋️  Train : {len(train_ds):,}")
print(f"   🔍 Val   : {len(val_ds):,}")
print("\n✅ Dataset sẵn sàng!")

📊 Cân bằng dataset (max 400/class):
   confidentiality     :  151  ███████
   dispute_resolution  :  374  ██████████████████
   insurance           :  400  ████████████████████
   liability           :  400  ████████████████████
   other               :  400  ████████████████████
   payment             :  400  ████████████████████
   termination         :  400  ████████████████████

   Total: 2,525 samples

   🏋️  Train : 2,272
   🔍 Val   : 253

✅ Dataset sẵn sàng!


---
## 🤖 Cell 8 — Cấu hình QLoRA và tải Qwen2.5-3B-Instruct

**QLoRA = 4-bit Quantization + LoRA adapters:**

| Config | Giá trị | Lý do |
|--------|---------|-------|
| `load_in_4bit` | True | Giảm 3B model từ ~6GB → ~2GB VRAM |
| `bnb_4bit_quant_type` | `nf4` | NormalFloat4 tốt hơn FP4 cho LLMs |
| `double_quant` | True | Giảm thêm ~0.4 bit/param |
| `lora_r` | 16 | Rank — cân bằng quality vs memory |
| `lora_alpha` | 32 | Scaling = 2×r (thực nghiệm tốt) |
| `target_modules` | 7 layers | Q/K/V/O + Gate/Up/Down proj |

**Ước tính VRAM trên P100 (16GB):**
- Model 4-bit: ~2 GB
- LoRA adapters: ~0.5 GB  
- Activations + optimizer: ~7 GB  
- **Total: ~10-11 GB** ✅

In [8]:
# ============================================================
# CELL 8: CẤU HÌNH QLORA VÀ TẢI MODEL
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# ── 8.1 BitsAndBytes 4-bit config ───────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# ── 8.2 LoRA config ─────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj",       # FFN (SwiGLU)
    ],
)

# ── 8.3 Tokenizer ────────────────────────────────────────────
print(f"⏳ Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"   pad_token: {tokenizer.pad_token}")
print(f"   vocab_size: {tokenizer.vocab_size:,}")

# ── 8.4 Model (4-bit) ────────────────────────────────────────
print(f"\n⏳ Loading model (4-bit): {MODEL_NAME}")
print("   (Có thể mất 2-3 phút để download ~6GB...)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

# ── 8.5 Chuẩn bị cho QLoRA ───────────────────────────────────
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)
model = get_peft_model(model, lora_config)
model.config.use_cache = False

# In trainable params
model.print_trainable_parameters()

# VRAM usage
used = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n💾 VRAM: {used:.2f} GB / {total:.1f} GB ({used/total*100:.1f}%)")
print("\n✅ Model + QLoRA sẵn sàng!")

⏳ Loading tokenizer: Qwen/Qwen2.5-3B-Instruct


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607

💾 VRAM: 1.55 GB / 15.6 GB (9.9%)

✅ Model + QLoRA sẵn sàng!


---
## 🏋️ Cell 9 — Training với SFTTrainer

**Hyperparameters tối ưu cho Kaggle P100/T4:**

| Parameter | Giá trị | Lý do |
|-----------|---------|-------|
| `batch_size` | 4 | P100 thoải mái hơn T4 |
| `grad_accum` | 4 | Effective batch = 16 |
| `max_seq_length` | 512 | Đủ cho contract clauses |
| `epochs` | 3 | Tránh overfitting |
| `lr` | 2e-4 | Best practice cho LoRA |
| `optimizer` | `paged_adamw_8bit` | Giảm VRAM optimizer states |
| `scheduler` | `cosine` | Tốt hơn linear cho LLM |

> ⏱️ Ước tính thời gian: **~25-40 phút** trên P100

In [9]:
# ============================================================
# CELL 9: CẤU HÌNH VÀ CHẠY SFTTRAINER
# ============================================================

# Kaggle lưu output vào /kaggle/working/
CKPT_DIR = "/kaggle/working/contract_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# P100 hỗ trợ float16 nhưng KHÔNG hỗ trợ bfloat16
# T4 cũng chỉ hỗ trợ float16
USE_FP16 = True

sft_config = SFTConfig(
    # Output
    output_dir=CKPT_DIR,

    # Epochs
    num_train_epochs=3,

    # Batch — P100 có thể dùng batch=4, T4 dùng batch=2
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,   # Effective batch = 16

    # Optimizer
    learning_rate=2e-4,
    weight_decay=0.01,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=50,
    max_grad_norm=0.3,

    # Memory
    gradient_checkpointing=True,
    fp16=USE_FP16,
    bf16=False,

    # Sequence
    max_seq_length=512,
    dataset_text_field="text",
    packing=False,

    # Logging
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # Tắt wandb (không cần trên Kaggle)
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

print("📋 Training Config:")
print(f"   Model            : {MODEL_NAME}")
print(f"   Train samples    : {len(train_ds):,}")
print(f"   Val samples      : {len(val_ds):,}")
print(f"   Epochs           : {sft_config.num_train_epochs}")
print(f"   Effective batch  : {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"   Learning rate    : {sft_config.learning_rate}")
print(f"   Max seq length   : {sft_config.max_seq_length}")
print(f"   Output dir       : {CKPT_DIR}")
print()

# ── Bắt đầu training ─────────────────────────────────────────
print("🚀 Bắt đầu training...")
print("   Ước tính: ~25-40 phút trên P100")
print("-" * 60)

train_result = trainer.train()

print("\n✅ Training hoàn tất!")
metrics = train_result.metrics
print(f"   Train loss  : {train_result.training_loss:.4f}")
print(f"   Total steps : {train_result.global_step}")
print(f"   Runtime     : {metrics.get('train_runtime', 0) / 60:.1f} phút")

Map:   0%|          | 0/2272 [00:00<?, ? examples/s]

Map:   0%|          | 0/253 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


📋 Training Config:
   Model            : Qwen/Qwen2.5-3B-Instruct
   Train samples    : 2,272
   Val samples      : 253
   Epochs           : 3
   Effective batch  : 16
   Learning rate    : 0.0002
   Max seq length   : 512
   Output dir       : /kaggle/working/contract_checkpoints

🚀 Bắt đầu training...
   Ước tính: ~25-40 phút trên P100
------------------------------------------------------------


Step,Training Loss,Validation Loss
100,0.904291,0.853106
200,0.434505,0.439196
300,0.230450,0.259247
400,0.147048,0.216651



✅ Training hoàn tất!
   Train loss  : 0.5455
   Total steps : 426
   Runtime     : 126.9 phút


---
## 📊 Cell 10 — Đánh giá model

Hai chiều đánh giá:
1. **Perplexity** (từ eval_loss) — metric tự động, không cần generate
2. **Category accuracy** — inference thực tế trên test cases

Thang đánh giá perplexity:
- `< 2.5` 🟢 Excellent
- `2.5 – 4.0` 🟡 Good
- `> 4.0` 🔴 Cần train thêm

In [10]:
# ============================================================
# CELL 10: ĐÁNH GIÁ MODEL
# ============================================================

# ── 10.1 Perplexity ─────────────────────────────────────────
print("⏳ Evaluating on validation set...")
eval_results = trainer.evaluate()

eval_loss  = eval_results.get("eval_loss", float("nan"))
perplexity = math.exp(eval_loss) if eval_loss < 20 else float("inf")

print(f"\n📊 Eval Loss  : {eval_loss:.4f}")
print(f"📊 Perplexity : {perplexity:.2f}")

if perplexity < 2.5:
    print("\n🟢 Excellent! Model học rất tốt.")
elif perplexity < 4.0:
    print("\n🟡 Good. Kết quả chấp nhận được.")
else:
    print("\n🔴 Cần train thêm hoặc tăng epochs/data.")


# ── 10.2 Inference function ──────────────────────────────────
def analyze_clause(text: str, max_new_tokens: int = 150) -> dict:
    """
    Phân tích một điều khoản hợp đồng.
    Input : str — nội dung điều khoản
    Output: dict {category, summary}
    """
    model.eval()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            "Analyze the following contract clause and return a JSON response:\n\n"
            f"CONTRACT CLAUSE:\n{text.strip()}"
        )},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    # Parse JSON
    valid_cats = {
        "salary", "payment", "confidentiality", "liability",
        "termination", "insurance", "dispute_resolution", "other"
    }
    try:
        m = re.search(r"\{[^{}]+\}", response, re.DOTALL)
        result = json.loads(m.group() if m else response)
        if result.get("category") not in valid_cats:
            result["category"] = "other"
        return result
    except (json.JSONDecodeError, AttributeError):
        return {"category": "other", "summary": response[:200]}


# ── 10.3 Test trên 5 ví dụ ──────────────────────────────────
TEST_CASES = [
    ("confidentiality",
     "The employee shall not disclose confidential information belonging to the company "
     "to any third party during or after the term of employment."),
    ("termination",
     "Either party may terminate this Agreement upon 30 days written notice. "
     "The Company may terminate immediately for cause."),
    ("liability",
     "IN NO EVENT SHALL EITHER PARTY BE LIABLE FOR INDIRECT OR CONSEQUENTIAL DAMAGES. "
     "TOTAL LIABILITY SHALL NOT EXCEED AMOUNTS PAID IN THE PRIOR TWELVE MONTHS."),
    ("dispute_resolution",
     "Any dispute arising from this Agreement shall be resolved by binding arbitration "
     "under AAA rules in New York, New York."),
    ("payment",
     "Client shall pay a monthly fee of $5,000 within 30 days of invoice. "
     "Late payments accrue interest at 1.5% per month."),
]

print("\n" + "=" * 65)
print("🔍 Category Accuracy Test")
print("=" * 65)

correct = 0
for expected, clause in TEST_CASES:
    result = analyze_clause(clause)
    pred   = result.get("category", "unknown")
    ok     = pred == expected
    if ok: correct += 1
    icon = "✅" if ok else "❌"
    print(f"\n{icon} Expected: [{expected:20s}] | Got: [{pred}]")
    print(f"   Summary: {result.get('summary', '')[:110]}")

acc = correct / len(TEST_CASES) * 100
print(f"\n{'=' * 65}")
print(f"📊 Accuracy: {correct}/{len(TEST_CASES)} ({acc:.0f}%)")

⏳ Evaluating on validation set...



📊 Eval Loss  : 0.2167
📊 Perplexity : 1.24

🟢 Excellent! Model học rất tốt.

🔍 Category Accuracy Test

✅ Expected: [confidentiality     ] | Got: [confidentiality]
   Summary: The Employee agrees that all Confidential Information is the sole property of the Company and that it will be 

✅ Expected: [termination         ] | Got: [termination]
   Summary: Either party may terminate this Agreement upon 30 days written notice.

✅ Expected: [liability           ] | Got: [liability]
   Summary: IN NO EVENT SHALL EITHER PARTY BE LIABLE FOR INDIRECT OR CONSEQUENTIAL DAMAGES.

✅ Expected: [dispute_resolution  ] | Got: [dispute_resolution]
   Summary: Any dispute arising from this Agreement shall be resolved by binding arbitration under AAA rules in New York, 

❌ Expected: [payment             ] | Got: [termination]
   Summary: This Agreement will be effective as of the date first above written and will continue in effect for three (3) 

📊 Accuracy: 4/5 (80%)


---
## 💾 Cell 11 — Lưu LoRA Adapter

Lưu vào `/kaggle/working/` — thư mục này được Kaggle giữ lại sau khi session kết thúc và có thể download về máy.

**Dung lượng:**
- LoRA adapter: ~50-100 MB *(chỉ các weight mới, không phải full model)*
- Tokenizer files: ~5 MB

**Download:** Output tab → `contract_lora_adapter` → Download

In [11]:
# ============================================================
# CELL 11: LƯU LORA ADAPTER
# ============================================================

ADAPTER_DIR = "/kaggle/working/contract_lora_adapter"
os.makedirs(ADAPTER_DIR, exist_ok=True)

# ── Lưu adapter + tokenizer ──────────────────────────────────
print(f"💾 Saving LoRA adapter → {ADAPTER_DIR}")
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# ── Lưu config training để tái hiện ─────────────────────────
training_config = {
    "base_model"      : MODEL_NAME,
    "lora_r"          : lora_config.r,
    "lora_alpha"      : lora_config.lora_alpha,
    "lora_dropout"    : lora_config.lora_dropout,
    "target_modules"  : list(lora_config.target_modules),
    "epochs"          : sft_config.num_train_epochs,
    "learning_rate"   : sft_config.learning_rate,
    "max_seq_length"  : sft_config.max_seq_length,
    "train_samples"   : len(train_ds),
    "categories"      : [
        "salary", "payment", "confidentiality", "liability",
        "termination", "insurance", "dispute_resolution", "other"
    ],
    "system_prompt"   : SYSTEM_PROMPT,
}
with open(f"{ADAPTER_DIR}/training_config.json", "w") as f:
    json.dump(training_config, f, indent=2, ensure_ascii=False)

# Liệt kê files đã lưu
print(f"\n📁 Files trong {ADAPTER_DIR}:")
for fname in sorted(os.listdir(ADAPTER_DIR)):
    size_mb = os.path.getsize(f"{ADAPTER_DIR}/{fname}") / 1024 / 1024
    print(f"   {fname:<45} {size_mb:6.2f} MB")

total_mb = sum(
    os.path.getsize(f"{ADAPTER_DIR}/{f}")
    for f in os.listdir(ADAPTER_DIR)
) / 1024 / 1024
print(f"\n   Total: {total_mb:.1f} MB")
print("\n✅ Lưu thành công!")
print("💡 Để download: Output tab (bên phải) → contract_lora_adapter → Download")

💾 Saving LoRA adapter → /kaggle/working/contract_lora_adapter

📁 Files trong /kaggle/working/contract_lora_adapter:
   README.md                                       0.00 MB
   adapter_config.json                             0.00 MB
   adapter_model.safetensors                     114.25 MB
   chat_template.jinja                             0.00 MB
   tokenizer.json                                 10.89 MB
   tokenizer_config.json                           0.00 MB
   training_config.json                            0.00 MB

   Total: 125.2 MB

✅ Lưu thành công!
💡 Để download: Output tab (bên phải) → contract_lora_adapter → Download


---
## 🔍 Cell 12 — Inference Pipeline (ContractAnalyzer)

Class tổng hợp hỗ trợ:
- `analyze(clause)` — phân tích một điều khoản
- `analyze_batch(clauses)` — phân tích nhiều điều khoản
- `load_from_adapter(path)` — load model từ adapter đã lưu *(cho session mới)*

In [12]:
# ============================================================
# CELL 12: INFERENCE PIPELINE — ContractAnalyzer
# ============================================================

class ContractAnalyzer:
    """
    Pipeline phân tích điều khoản hợp đồng.

    Ví dụ sử dụng:
    ---------------
    # Dùng model từ session hiện tại
    analyzer = ContractAnalyzer()

    # Phân tích một điều khoản
    result = analyzer.analyze("The employee shall not disclose...")
    print(result)
    # → {"category": "confidentiality", "summary": "..."}

    # Phân tích nhiều điều khoản
    results = analyzer.analyze_batch([clause1, clause2, clause3])
    """

    VALID_CATEGORIES = {
        "salary", "payment", "confidentiality", "liability",
        "termination", "insurance", "dispute_resolution", "other",
    }

    SYSTEM_PROMPT = (
        "You are a legal contract analysis expert. "
        "Analyze the given contract clause and return a JSON object with:\n"
        "- \"category\": one of [salary, payment, confidentiality, liability, "
        "termination, insurance, dispute_resolution, other]\n"
        "- \"summary\": a concise 1-2 sentence summary of the clause\n\n"
        "Return ONLY valid JSON, no additional text."
    )

    def __init__(self, model=None, tokenizer=None):
        """
        Args:
            model     : Model đã load. None = dùng global `model`.
            tokenizer : Tokenizer. None = dùng global `tokenizer`.
        """
        import builtins
        g = vars(builtins)

        self.model     = model     or globals().get("model")
        self.tokenizer = tokenizer or globals().get("tokenizer")

        if self.model is None or self.tokenizer is None:
            raise ValueError(
                "Không tìm thấy model/tokenizer. "
                "Chạy Cell 8 trước hoặc dùng load_from_adapter()."
            )
        self.model.eval()
        print("✅ ContractAnalyzer khởi tạo thành công.")

    @classmethod
    def load_from_adapter(
        cls,
        adapter_path: str = "/kaggle/working/contract_lora_adapter",
        base_model: str   = "Qwen/Qwen2.5-3B-Instruct",
    ) -> "ContractAnalyzer":
        """
        Load model từ adapter đã lưu (dùng cho session mới).

        Args:
            adapter_path : Đường dẫn thư mục chứa adapter.
            base_model   : Tên base model trên HuggingFace.
        """
        print(f"⏳ Loading base model: {base_model}")
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        tok = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token

        base = AutoModelForCausalLM.from_pretrained(
            base_model,
            quantization_config=bnb,
            device_map="auto",
            trust_remote_code=True,
        )
        print(f"⏳ Loading LoRA adapter: {adapter_path}")
        mdl = PeftModel.from_pretrained(base, adapter_path)
        print("✅ Loaded!")
        return cls(model=mdl, tokenizer=tok)

    def analyze(self, clause_text: str, max_new_tokens: int = 150) -> dict:
        """
        Phân tích một điều khoản hợp đồng.

        Args:
            clause_text    : Nội dung điều khoản.
            max_new_tokens : Số token tối đa sinh ra.

        Returns:
            dict: {"category": str, "summary": str}
        """
        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user",   "content": (
                "Analyze the following contract clause and return a JSON response:\n\n"
                f"CONTRACT CLAUSE:\n{clause_text.strip()}"
            )},
        ]
        prompt = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=512
        ).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.1,
                do_sample=True,
                repetition_penalty=1.1,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        gen = outputs[0][inputs["input_ids"].shape[-1]:]
        response = self.tokenizer.decode(gen, skip_special_tokens=True).strip()
        return self._parse(response)

    def analyze_batch(self, clauses: list, verbose: bool = True) -> list:
        """
        Phân tích danh sách các điều khoản.

        Args:
            clauses : List[str] — danh sách điều khoản.
            verbose : In progress.

        Returns:
            List[dict]
        """
        results = []
        for i, clause in enumerate(clauses, 1):
            result = self.analyze(clause)
            results.append(result)
            if verbose:
                print(f"[{i:2d}/{len(clauses)}] [{result['category']:20s}] "
                      f"{result['summary'][:80]}")
        return results

    def _parse(self, response_text: str) -> dict:
        try:
            m = re.search(r"\{[^{}]+\}", response_text, re.DOTALL)
            result = json.loads(m.group() if m else response_text)
            if result.get("category") not in self.VALID_CATEGORIES:
                result["category"] = "other"
            result.setdefault("summary", response_text[:200])
            return result
        except (json.JSONDecodeError, AttributeError):
            return {"category": "other", "summary": response_text[:200]}


# Khởi tạo analyzer từ session hiện tại
analyzer = ContractAnalyzer()

✅ ContractAnalyzer khởi tạo thành công.


---
## 🧪 Cell 13 — Demo Inference

Kiểm tra model với 6 loại điều khoản khác nhau và ví dụ chính từ yêu cầu.

In [13]:
# ============================================================
# CELL 13: DEMO INFERENCE
# ============================================================

# Ví dụ chính từ yêu cầu
MAIN_EXAMPLE = (
    "The employee shall not disclose confidential information "
    "belonging to the company to any third party."
)

print("=" * 65)
print("📄 VÍ DỤ CHÍNH")
print("=" * 65)
print(f"\nInput:\n  {MAIN_EXAMPLE}\n")
result = analyzer.analyze(MAIN_EXAMPLE)
print("Output:")
print(json.dumps(result, indent=2, ensure_ascii=False))

# Batch demo
DEMO_CLAUSES = [
    "The employee shall not disclose confidential information to any third party.",
    "The annual base salary shall be USD 120,000 paid in bi-weekly installments.",
    "Client shall pay a monthly retainer of $3,000 due on the first business day.",
    "This Agreement terminates automatically upon expiration of the Term unless "
    "renewed in writing at least 30 days before expiration.",
    "Contractor shall maintain general liability insurance with coverage of "
    "not less than $1,000,000 per occurrence.",
    "All disputes shall be resolved through binding arbitration under JAMS rules "
    "in San Francisco, California.",
]

print("\n" + "=" * 65)
print("📋 BATCH DEMO — 6 điều khoản")
print("=" * 65 + "\n")

batch_results = analyzer.analyze_batch(DEMO_CLAUSES)

print("\n" + "=" * 65)
print("📊 TỔNG HỢP")
print("=" * 65)
for i, (clause, res) in enumerate(zip(DEMO_CLAUSES, batch_results), 1):
    print(f"\n{i}. [{res['category'].upper()}]")
    print(f"   In : {clause[:70]}...")
    print(f"   Out: {res['summary'][:100]}")

print("\n✅ Demo hoàn tất!")

📄 VÍ DỤ CHÍNH

Input:
  The employee shall not disclose confidential information belonging to the company to any third party.

Output:
{
  "category": "confidentiality",
  "summary": "In addition, the Employee agrees that all Confidential Information is the property of the Company and that it will be used solely in connection with the business activities of the Company."
}

📋 BATCH DEMO — 6 điều khoản

[ 1/6] [confidentiality     ] In addition, in the event that an assignment or merger is completed pursuant to 
[ 2/6] [termination         ] Upon expiration of this Agreement or upon any termination hereof, the Company wi
[ 3/6] [termination         ] This Agreement will commence on the date hereof and continue in full force and e
[ 4/6] [termination         ] This Agreement terminates automatically upon expiration of the Term unless renew
[ 5/6] [insurance           ] Contractor shall maintain general liability insurance with coverage of not less 
[ 6/6] [dispute_resolution  ] All dispu

---
## 💬 Cell 14 — Interactive Input (Tùy chọn)

Nhập điều khoản hợp đồng thủ công và nhận phân tích real-time.

> Gõ `quit` để thoát khỏi vòng lặp.

In [14]:
# ============================================================
# CELL 14: INTERACTIVE INPUT (TÙY CHỌN)
# ============================================================
# Bỏ comment để chạy chế độ tương tác

# print("🔍 CONTRACT CLAUSE ANALYZER — Interactive Mode")
# print("Gõ 'quit' để thoát.\n")
#
# while True:
#     clause = input("📝 Nhập điều khoản: ").strip()
#     if clause.lower() in ("quit", "exit", "q"):
#         print("👋 Thoát.")
#         break
#     if not clause:
#         print("⚠️  Vui lòng nhập nội dung.\n")
#         continue
#     print("⏳ Đang phân tích...")
#     result = analyzer.analyze(clause)
#     print("\n📊 KẾT QUẢ:")
#     print(json.dumps(result, indent=2, ensure_ascii=False))
#     print()

print("💡 Bỏ comment đoạn trên để dùng chế độ interactive.")
print("   Hoặc gọi trực tiếp: analyzer.analyze('điều khoản của bạn')")

💡 Bỏ comment đoạn trên để dùng chế độ interactive.
   Hoặc gọi trực tiếp: analyzer.analyze('điều khoản của bạn')


---

## ✅ Tóm tắt Phase 1

| Bước | Mô tả | Status |
|------|--------|:------:|
| Cài đặt thư viện | peft, trl, bitsandbytes | ✅ |
| Load CUAD dataset | theatticusproject/cuad-qa | ✅ |
| Category mapping | Keyword → 8 categories | ✅ |
| Instruction format | Qwen2.5 chat template | ✅ |
| Dataset cân bằng | Cap 400/class, 90/10 split | ✅ |
| QLoRA config | 4-bit NF4, LoRA r=16 | ✅ |
| SFTTrainer | 3 epochs, cosine LR | ✅ |
| Đánh giá | Perplexity + accuracy | ✅ |
| Lưu adapter | `/kaggle/working/` | ✅ |
| Inference pipeline | ContractAnalyzer class | ✅ |

### 🚀 Phase 2 (tiếp theo)
- **DPO/RLHF**: Cải thiện output quality
- **Tiếng Việt**: Fine-tune thêm với điều khoản tiếng Việt  
- **FastAPI**: Đóng gói thành REST API  
- **ROUGE/BLEU**: Đánh giá chất lượng summary

### 🔗 Tài nguyên
- [CUAD Dataset](https://huggingface.co/datasets/theatticusproject/cuad-qa)
- [Qwen2.5-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct)
- [PEFT Docs](https://huggingface.co/docs/peft)
- [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer)